# 01 · SQL analysis
Reads the tables and views built by `python src/run_sql.py` (which runs `sql/01_schema.sql` … `sql/08_views.sql` against `data/grid.duckdb`).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
con = duckdb.connect("data/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
q = lambda sql: con.execute(sql).df()

## Row counts

In [2]:
tables = ["raw_demand", "raw_generation", "raw_weather", "dim_fuel", "dim_calendar",
          "fact_load", "fact_generation", "fact_weather", "model_hourly"]
pd.DataFrame({"table": tables, "rows": [q(f"SELECT COUNT(*) n FROM {t}").n[0] for t in tables]})

,table,rows
0,raw_demand,61367
1,raw_generation,449006
2,raw_weather,245472
3,dim_fuel,9
4,dim_calendar,61368
5,fact_load,61368
6,fact_generation,449006
7,fact_weather,245472
8,model_hourly,61368


## Data quality (sql/02_quality.sql)

In [3]:
q("SELECT * FROM qa_summary")

,check_name,n
0,missing demand hours,1
1,missing generation hours,24
2,duplicate rows,0
3,"demand < 0 or > 100,000 MWh",0
4,negative wind/solar,0
5,gas hours misfiled as OTH,122


In [4]:
q("SELECT * FROM qa_missing_demand")

/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_24793/681562206.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()


,period,local_ts
0,2021-11-07 06:00:00+00:00,2021-11-07 01:00:00


In [5]:
q("""SELECT CAST(local_ts AS DATE) AS local_date, COUNT(*) AS hours
     FROM qa_missing_generation GROUP BY 1""")

,local_date,hours
0,2025-12-05,23
1,2025-12-06,1


In [6]:
q("""SELECT CAST(period AS DATE) AS utc_date, COUNT(*) AS hours, ROUND(AVG(value)) AS avg_oth_mwh
     FROM qa_outliers WHERE issue LIKE 'OTH spike%' GROUP BY 1 ORDER BY 1""")

,utc_date,hours,avg_oth_mwh
0,2019-12-18,17,23235.0
1,2019-12-19,7,22092.0
2,2020-01-08,17,11342.0
3,2020-01-09,7,8802.0
4,2020-01-15,17,13991.0
5,2020-01-16,7,15989.0
6,2020-01-22,17,15883.0
7,2020-01-23,7,21564.0
8,2020-03-18,18,15679.0
9,2020-03-19,6,15979.0


## Window functions (sql/04_windows.sql)

In [7]:
q("SELECT * FROM peak_growth_yearly")

,year,peak_mwh,peak_change_mwh,peak_growth_pct
0,2019,74533.0,NaN,NaN
1,2020,74166.0,-367.0,-0.49
2,2021,73476.0,-690.0,-0.93
3,2022,79830.0,6354.0,8.65
4,2023,85432.0,5602.0,7.02
5,2024,85544.0,112.0,0.13
6,2025,83597.0,-1947.0,-2.28


In [8]:
q("SELECT year, local_ts, demand_mwh, ROUND(avg_temp_c,1) AS avg_temp_c FROM top_peaks_yearly WHERE peak_rank <= 3")

,year,local_ts,demand_mwh,avg_temp_c
0,2019,2019-08-12 17:00:00,74533.0,37.8
1,2019,2019-08-12 18:00:00,74457.0,37.5
2,2019,2019-08-26 17:00:00,74338.0,37.1
3,2020,2020-08-13 17:00:00,74166.0,37.9
4,2020,2020-08-28 16:00:00,74104.0,37.2
5,2020,2020-08-14 17:00:00,74091.0,38.3
6,2021,2021-08-31 17:00:00,73476.0,33.8
7,2021,2021-08-24 17:00:00,73471.0,34.8
8,2021,2021-08-25 17:00:00,73364.0,33.1
9,2022,2022-07-20 17:00:00,79830.0,38.7


In [9]:
# Spot check: lag and ramp columns vs raw data for three hours
q("""SELECT period, demand_mwh, demand_lag_24, demand_lag_168, ramp_1h, demand_7d_avg
     FROM model_hourly
     WHERE period IN ('2021-02-15 06:00+00', '2023-08-10 23:00+00', '2025-06-01 18:00+00')""")

/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_24793/681562206.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()


,period,demand_mwh,demand_lag_24,demand_lag_168,ramp_1h,demand_7d_avg
0,2021-02-15 06:00:00+00:00,65766.0,55867.0,36392.0,-923.0,52477.470238
1,2023-08-10 23:00:00+00:00,85432.0,83961.0,83288.0,277.0,69575.934524
2,2025-06-01 18:00:00+00:00,62636.0,60845.0,68065.0,3324.0,56461.714286


In [10]:
q("""SELECT period, demand_mwh FROM raw_demand
     WHERE period IN ('2023-08-10 22:00+00','2023-08-09 23:00+00','2023-08-03 23:00+00','2023-08-10 23:00+00')
     ORDER BY period""")

/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_24793/681562206.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()


,period,demand_mwh
0,2023-08-03 23:00:00+00:00,83288
1,2023-08-09 23:00:00+00:00,83961
2,2023-08-10 22:00:00+00:00,85155
3,2023-08-10 23:00:00+00:00,85432


## Fuel mix and solar (sql/05_fuel_mix.sql)

In [11]:
q("SELECT * FROM fuel_share_yearly")

,year,gas_pct,coal_pct,nuclear_pct,wind_pct,solar_pct,hydro_other_pct,total_twh
0,2019,47.4,20.4,10.8,20.0,1.0,0.3,383.1
1,2020,45.6,18.1,11.0,22.9,2.2,0.2,379.7
2,2021,41.9,19.2,10.3,24.4,3.9,0.3,391.3
3,2022,42.6,16.8,9.8,25.0,5.5,0.3,429.0
4,2023,45.2,14.0,9.2,24.2,7.1,0.3,446.0
5,2024,44.1,12.7,8.4,24.1,10.3,0.5,463.9
6,2025,40.9,13.0,8.6,23.5,13.8,0.2,487.7


In [12]:
q("SELECT * FROM midday_solar_share")

,year,midday_solar_pct,avg_midday_solar_mwh,max_solar_mwh
0,2019,2.5,1191.0,1803.0
1,2020,5.1,2358.0,3824.0
2,2021,9.2,4406.0,6986.0
3,2022,13.0,6785.0,10039.0
4,2023,17.1,9256.0,13639.0
5,2024,24.7,13898.0,21589.0
6,2025,32.9,19832.0,29503.0


## Net load and the duck curve (sql/06_net_load.sql)

In [13]:
q("SELECT * FROM evening_ramp_yearly")

,year,avg_daily_max_ramp_mwh,max_ramp_mwh
0,2019,3063.0,13369.0
1,2020,3249.0,13901.0
2,2021,3838.0,15472.0
3,2022,5396.0,19049.0
4,2023,6338.0,22773.0
5,2024,9654.0,26681.0
6,2025,13756.0,30132.0


In [14]:
q("SELECT * FROM top_evening_ramps WHERE ramp_rank <= 2")

,year,local_ts,net_load_mwh,ramp_3h_mwh,ramp_rank
0,2019,2019-01-19 19:00:00,37132.0,13369.0,1
1,2019,2019-12-29 19:00:00,33586.0,12346.0,2
2,2020,2020-12-24 19:00:00,34485.0,13901.0,1
3,2020,2020-02-26 20:00:00,44589.0,13835.0,2
4,2021,2021-02-06 19:00:00,33337.0,15472.0,1
5,2021,2021-10-29 20:00:00,35052.0,15330.0,2
6,2022,2022-03-24 21:00:00,39299.0,19049.0,1
7,2022,2022-12-14 19:00:00,41178.0,17607.0,2
8,2023,2023-12-29 19:00:00,45483.0,22773.0,1
9,2023,2023-12-27 19:00:00,44878.0,22545.0,2


## Winter Storm Uri (sql/07_uri.sql)

In [15]:
q("SELECT * FROM uri_daily")

,local_date,min_temp_c,peak_demand_mwh,avg_demand_mwh,avg_base_demand_mwh,avg_gas_mwh,avg_base_gas_mwh,avg_wind_mwh,avg_base_wind_mwh,avg_nuclear_mwh,missing_hours
0,2021-02-13,-2.3,63694.0,58720.0,37497.0,37191.0,10094.0,4034.0,12708.0,5138.0,0
1,2021-02-14,-7.4,69215.0,62018.0,37506.0,37092.0,9154.0,7237.0,13275.0,5139.0,0
2,2021-02-15,-12.0,65766.0,50152.0,39256.0,32202.0,13532.0,3223.0,10690.0,4156.0,0
3,2021-02-16,-13.9,46846.0,45055.0,40900.0,28919.0,17457.0,3695.0,7814.0,3784.0,0
4,2021-02-17,-4.0,49252.0,45160.0,42436.0,29810.0,17717.0,2546.0,9446.0,3793.0,0
5,2021-02-18,-4.8,55426.0,51696.0,45534.0,31597.0,20281.0,5812.0,10261.0,4575.0,0
6,2021-02-19,-8.2,58410.0,49153.0,46882.0,27003.0,21009.0,6007.0,10155.0,5142.0,0
7,2021-02-20,-3.1,52556.0,42883.0,37497.0,18137.0,10094.0,10660.0,12708.0,5142.0,0


In [16]:
u = q("SELECT * FROM uri_hourly")
peak = u.loc[u.demand_mwh.idxmax()]
after = u[(u.local_ts > peak.local_ts) & (u.local_date <= pd.Timestamp('2021-02-18'))]  # outage window Feb 15-18
trough = after.loc[after.total_gen_mwh.idxmin()]
uri = {
    "peak demand hour (local)": peak.local_ts,
    "peak served demand MWh": peak.demand_mwh,
    "total generation at peak MWh": peak.total_gen_mwh,
    "lowest generation during outages (local)": trough.local_ts,
    "lowest generation MWh": trough.total_gen_mwh,
    "generation drop MWh": peak.total_gen_mwh - trough.total_gen_mwh,
    "generation drop %": round(100 * (peak.total_gen_mwh - trough.total_gen_mwh) / peak.total_gen_mwh, 1),
    "gas at peak MWh": peak.gas_mwh, "gas at trough MWh": trough.gas_mwh,
    "coal at peak MWh": peak.coal_mwh, "coal at trough MWh": trough.coal_mwh,
    "nuclear at peak MWh": peak.nuclear_mwh, "nuclear at trough MWh": trough.nuclear_mwh,
    "wind at peak MWh": peak.wind_mwh, "wind at trough MWh": trough.wind_mwh,
    "temp at trough C": round(trough.avg_temp_c, 1),
    "storm-week wind vs baseline %": round(100 * (u.wind_mwh.mean() / u.base_wind.mean() - 1), 1),
    "missing hours in storm week": int(u.demand_mwh.isna().sum() + u.total_gen_mwh.isna().sum()),
}
pd.Series(uri, dtype=object)

/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_24793/681562206.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()


peak demand hour (local)                    2021-02-14 20:00:00
peak served demand MWh                                  69215.0
total generation at peak MWh                            68022.0
lowest generation during outages (local)    2021-02-17 02:00:00
lowest generation MWh                                   41710.0
generation drop MWh                                     26312.0
generation drop %                                          38.7
gas at peak MWh                                         43798.0
gas at trough MWh                                       28216.0
coal at peak MWh                                        10829.0
coal at trough MWh                                       6274.0
nuclear at peak MWh                                      5140.0
nuclear at trough MWh                                    3781.0
wind at peak MWh                                         8087.0
wind at trough MWh                                       3315.0
temp at trough C                        

## Views and performance (sql/08_views.sql)

In [17]:
for v in ["daily_peak", "fuel_mix_yearly", "net_load_hourly"]:
    display(q(f"SELECT * FROM {v} LIMIT 5"))

,local_date,year,month,season,peak_demand_mwh,avg_demand_mwh,max_temp_c,min_temp_c
0,2019-01-06,2019,1,Winter,37353.0,35008.791667,20.300,7.725
1,2019-03-17,2019,3,Spring,37678.0,34508.041667,17.250,7.150
2,2019-05-06,2019,5,Spring,49084.0,41298.958333,25.850,19.075
3,2019-05-18,2019,5,Spring,45787.0,42145.333333,27.025,22.425
4,2019-05-21,2019,5,Spring,54182.0,45785.750000,29.400,22.550


,year,fueltype,fuel_name,is_renewable,generation_mwh,share_pct
0,2024,NG,Natural gas,False,204366166.0,44.057153
1,2024,WAT,Hydro,True,423905.0,0.091385
2,2024,OTH,Other,False,1849052.0,0.398618
3,2024,UES,Unknown energy storage,False,-706122.0,-0.152225
4,2024,SUN,Solar,True,47748993.0,10.293703


/var/folders/q1/96vrq0fx6vvgxg_ntnn8dhx80000gn/T/ipykernel_24793/681562206.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  q = lambda sql: con.execute(sql).df()


,period,local_ts,local_date,local_hour,year,demand_mwh,wind_mwh,solar_mwh,net_load_mwh,ramp_3h_mwh
0,2019-01-01 00:00:00+00:00,2018-12-31 18:00:00,2018-12-31,18,2018,38397.0,5020.0,161.0,33216.0,NaN
1,2019-01-01 01:00:00+00:00,2018-12-31 19:00:00,2018-12-31,19,2018,40387.0,5597.0,0.0,34790.0,NaN
2,2019-01-01 02:00:00+00:00,2018-12-31 20:00:00,2018-12-31,20,2018,39769.0,6637.0,0.0,33132.0,NaN
3,2019-01-01 03:00:00+00:00,2018-12-31 21:00:00,2018-12-31,21,2018,39016.0,8692.0,0.0,30324.0,-2892.0
4,2019-01-01 04:00:00+00:00,2018-12-31 22:00:00,2018-12-31,22,2018,38473.0,11110.0,0.0,27363.0,-7427.0


In [18]:
import re
def explain_ms(sql):
    plan = con.execute("EXPLAIN ANALYZE " + sql).fetchall()[0][1]
    return float(re.search(r"Total Time: ([0-9.]+)s", plan).group(1)) * 1000

rolling = """SELECT period, AVG(demand_mwh) OVER (ORDER BY period ROWS BETWEEN 167 PRECEDING AND CURRENT ROW)
             FROM model_hourly"""
uri_self_join = open("sql/07_uri.sql").read().split("CREATE OR REPLACE TABLE uri_hourly AS")[1].split(";")[0]
timings = pd.DataFrame({
    "query": ["rolling 7-day average (61k rows)", "Uri baseline self-join"],
    "median_ms_of_5": [np.median([explain_ms(s) for _ in range(5)]) for s in (rolling, uri_self_join)],
}).round(1)
timings

,query,median_ms_of_5
0,rolling 7-day average (61k rows),4.2
1,Uri baseline self-join,1.7
